# Notebook 7 - Visualize the predicitions made on the Fingerplan neighborhoods

This notebook makes a visualization of the prediction made on all neighborhoods in the Finger plan

The map can be found in /results/figures/fingerplan_predict_map.html and can be viewed in a browser


In [38]:
# Imports and paths
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import leafmap.foliumap as leafmap

# Prefer data/interim predictions; fallback to results/models
pred_paths = [
    #os.path.join('..', 'data', 'interim', 'finger_predictions.csv'),
    os.path.join('..', 'results', 'models', 'finger_predictions.csv'),
]
#shp_path = os.path.join('..', 'data', 'raw', 'denmark_shapefile', 'clus_denmark_lesstnan_50.shp')
shp_path = os.path.join('..', 'data', 'raw', 'fingerplanen_shapefile', 'GTD_clus_fingerplan_lessthan_50.shp')
map_out_dir = os.path.join('..', 'results', 'figures')
map_out_path = os.path.join(map_out_dir, 'fingerplan_predict_map.html')

print('Predictions path candidates:', pred_paths)
print('Shapefile path:', shp_path)
print('Output HTML:', map_out_path)


Predictions path candidates: ['../results/models/finger_predictions.csv']
Shapefile path: ../data/raw/fingerplanen_shapefile/GTD_clus_fingerplan_lessthan_50.shp
Output HTML: ../results/figures/fingerplan_predict_map.html


In [39]:
# Load predictions and shapefile, normalize join keys, merge
# Load predictions with fallback
pred_path = None
for p in pred_paths:
    if os.path.exists(p):
        pred_path = p
        break
if pred_path is None:
    raise FileNotFoundError(f"Could not find finger_predictions.csv in: {pred_paths}")

pred_df = pd.read_csv(pred_path)

# Ensure prediction column exists
if 'prediction' not in pred_df.columns:
    raise KeyError("'prediction' column not found in predictions CSV.")

# Normalize cluster id in predictions
if 'Cluster_id' in pred_df.columns:
    pred_df = pred_df.rename(columns={'Cluster_id': 'cluster_id'})
elif 'cluster_id' not in pred_df.columns:
    # Heuristic: look for columns containing cluster/munic
    heur = [c for c in pred_df.columns if ('cluster' in c.lower()) or ('munic' in c.lower())]
    if heur:
        pred_df = pred_df.rename(columns={heur[0]: 'cluster_id'})
    else:
        raise KeyError("No cluster_id column found in predictions.")

# Load shapefile
gdf = gpd.read_file(shp_path)

# Detect/normalize shapefile join key (prefer 'munic_clus')
join_col = None
if 'munic_clus' in gdf.columns:
    join_col = 'munic_clus'
elif any(c.lower() == 'cluster_id' for c in gdf.columns):
    join_col = [c for c in gdf.columns if c.lower() == 'cluster_id'][0]
else:
    # Prefer columns containing both 'munic' and 'clus'
    both = [c for c in gdf.columns if ('munic' in c.lower()) and ('clus' in c.lower())]
    if both:
        join_col = both[0]
    else:
        # Fallback to generic cluster/munic heuristics
        heur = [c for c in gdf.columns if ('cluster' in c.lower()) or ('munic' in c.lower())]
        if heur:
            join_col = heur[0]

if join_col is None:
    raise KeyError("Could not detect join key in shapefile; expected 'munic_clus' or similar.")

if join_col != 'cluster_id':
    gdf = gdf.rename(columns={join_col: 'cluster_id'})

# Align dtypes for merge and clean ids
pred_df['cluster_id'] = pred_df['cluster_id'].astype(str).str.strip()
gdf['cluster_id'] = gdf['cluster_id'].astype(str).str.strip()

# Also create cleaned version without leading zeros to improve matching
pred_df['cluster_id_clean'] = pred_df['cluster_id'].str.lstrip('0')
gdf['cluster_id_clean'] = gdf['cluster_id'].str.lstrip('0')

# Reproject to EPSG:4326 (web mercator friendly long/lat)
try:
    if gdf.crs is None:
        print('WARNING: shapefile has no CRS; proceeding without reprojection.')
    else:
        gdf = gdf.to_crs(epsg=4326)
except Exception as e:
    print('WARNING: reprojection failed:', e)

# Merge using cleaned ids
gdf_merged = gdf.merge(pred_df[['cluster_id_clean', 'prediction']], left_on='cluster_id_clean', right_on='cluster_id_clean', how='left')

# Coerce prediction dtype (numeric) and compute counts using 'prediction' column
gdf_merged['prediction'] = pd.to_numeric(gdf_merged['prediction'], errors='coerce')

# Create readable labels and a status
label_map = {0: 'Class 0', 1: 'Class 1', 2: 'Class 2'}
gdf_merged['prediction_label'] = gdf_merged['prediction'].map(label_map)
gdf_merged['status'] = np.where(gdf_merged['prediction'].isna(), 'NoPrediction', 'Predicted')

rows_total = len(gdf_merged)
with_pred = int(gdf_merged['prediction'].notna().sum())
without_pred = int(gdf_merged['prediction'].isna().sum())

print('Rows:', rows_total)
print('With predictions:', with_pred)
print('Without predictions:', without_pred)

Rows: 4020
With predictions: 4020
Without predictions: 0


In [40]:
# Build Leafmap and save HTML
# Prepare GeoJSON
import json
geojson = json.loads(gdf_merged.to_json())

# Colors for classes and no prediction
class_colors = {0: '#2ca02c', 1: '#1f77b4', 2: '#ff7f0e'}
no_pred_color = '#7f7f7f'

# Style function per feature (cast prediction to int for exact color mapping)
def style_function(feature):
    pred_raw = feature['properties'].get('prediction')
    pred_val = None
    try:
        if pred_raw is None:
            pred_val = None
        else:
            # Handle NaN values safely
            if isinstance(pred_raw, float) and np.isnan(pred_raw):
                pred_val = None
            else:
                pred_val = int(float(pred_raw))
    except Exception:
        pred_val = None

    color = class_colors.get(pred_val, no_pred_color)
    return {
        'fillColor': color,
        'color': '#555555',
        'weight': 0.5,
        'fillOpacity': 0.6,
    }

hover_style = {
    'fillOpacity': 0.8,
    'weight': 1.0,
}

# Create map centered roughly around Copenhagen
m = leafmap.Map(center=(55.6761, 12.5683), zoom=10)
m.add_geojson(
    geojson,
    layer_name='Fingerplan Predictions',
    style_function=style_function,
    hover_style=hover_style,
    info_mode='on_hover'
)

# Legend
m.add_legend(
    title='Prediction Classes',
    labels=['Class 0', 'Class 1', 'Class 2', 'NoPrediction'],
    colors=['#2ca02c', '#1f77b4', '#ff7f0e', '#7f7f7f']
)

# Save HTML
os.makedirs(map_out_dir, exist_ok=True)
try:
    m.to_html(map_out_path)
except Exception:
    # Fallback for environments where to_html isn't available
    m.save(map_out_path)

print(f"Saved map to: {map_out_path}")


Saved map to: ../results/figures/fingerplan_predict_map.html


### Output
The interactive map is saved to `../results/figures/fingerplan_predict_map.html`. Open it in your browser for a full-screen view.


In [41]:
# Export GeoPackage for GIS consumption
import os

# Use the same output directory as the HTML map
gpkg_out_dir = map_out_dir
gpkg_out_path = os.path.join(gpkg_out_dir, 'fingerplan_predictions.gpkg')
#gpkg_out_path = os.path.join(gpkg_out_dir, 'DK_predictions.gpkg')


os.makedirs(gpkg_out_dir, exist_ok=True)

try:
    # Ensure CRS is set before export
    if gdf_merged.crs is None:
        print('WARNING: GeoDataFrame has no CRS; setting EPSG:4326 for export.')
        gdf_merged = gdf_merged.set_crs(epsg=4326)

    # Drop reserved ID fields that conflict with GeoPackage FID
    drop_cols = [c for c in gdf_merged.columns if c.lower() in ('fid', 'ogc_fid')]
    gdf_to_save = gdf_merged.drop(columns=drop_cols, errors='ignore').reset_index(drop=True)

    # Save to GeoPackage with a clear layer name
    gdf_to_save.to_file(gpkg_out_path, layer='fingerplan_predictions', driver='GPKG')
    print(f"Saved GeoPackage to: {gpkg_out_path}")
except Exception as e:
    print('ERROR: failed to save GeoPackage:', e)


Saved GeoPackage to: ../results/figures/fingerplan_predictions.gpkg
